In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/Master/LTAINC")
PROJECT_DIR = DRIVE_ROOT / "lightweight-medical-model"

In [ ]:
%cd {PROJECT_DIR}
# Bỏ comment dòng dưới nếu cần lấy phiên bản code mới nhất.
# !git pull origin main

# Grad-CAM cho MK-MNet

Notebook sinh Grad-CAM từ checkpoint MK-MNet width 0.25 và lưu ảnh thử nghiệm riêng trong `gradcam-results/`.

In [ ]:
import torch

CHECKPOINT_RELATIVE = Path("outputs/busi/mk_mnet/img_224/width_0.25/oversample_1/lambda_0.8/lr_0.0003/weight_decay_0.0001/patience_10/best_model.pt")
ARTIFACT_CANDIDATES = [
    PROJECT_DIR,
    DRIVE_ROOT / "colab-result/lightweight-medical-model",
    PROJECT_DIR / "colab-result/lightweight-medical-model",
]

ARTIFACT_ROOT = next(
    (root for root in ARTIFACT_CANDIDATES if (root / CHECKPOINT_RELATIVE).is_file()),
    None,
)
if ARTIFACT_ROOT is None:
    checked = "\n".join(f"- {(root / CHECKPOINT_RELATIVE)}" for root in ARTIFACT_CANDIDATES)
    raise FileNotFoundError(f"Không tìm thấy checkpoint. Đã kiểm tra:\n{checked}")

CHECKPOINT = ARTIFACT_ROOT / CHECKPOINT_RELATIVE
DATASET_DIR = ARTIFACT_ROOT / "data/busi"
STAGE = 5
TARGET_CLASS = "true"
OUTPUT_DIR = PROJECT_DIR / f"gradcam-results/mk_mnet_width025/stage_{STAGE}/target_{TARGET_CLASS}"

assert DATASET_DIR.is_dir(), f"Không tìm thấy dataset: {DATASET_DIR}"
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Code:", PROJECT_DIR)
print("Artifacts:", ARTIFACT_ROOT)
print("Checkpoint:", CHECKPOINT)
print("Output:", OUTPUT_DIR)

In [ ]:
!python generate_gradcam.py \
  --model mk_mnet \
  --width-mult 0.25 \
  --checkpoint "$CHECKPOINT" \
  --dataset-dir "$DATASET_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --stage $STAGE \
  --target-class $TARGET_CLASS \
  --num-samples 9 \
  --seed 42

## Xem kết quả

In [ ]:
from pathlib import Path
from IPython.display import Image, display

result_paths = sorted(Path(OUTPUT_DIR).glob("*.png"))
print(f"Đã tạo {len(result_paths)} ảnh trong {OUTPUT_DIR}")
for result_path in result_paths:
    display(Image(filename=str(result_path)))

## Chạy toàn bộ test set cho tất cả model

Cell dưới chạy tuần tự các model có checkpoint. Model thiếu checkpoint sẽ được báo và bỏ qua.

In [ ]:
import subprocess
import sys

MODEL_RUNS = [
    {
        "name": "mednet",
        "model": "mednet",
        "checkpoints": [
            "outputs/busi/classification/img_224/cbam/run_1/best_model.pt",
            "outputs/busi/classification_cu/img_224/cbam/run_1/best_model.pt",
        ],
    },
    {
        "name": "multitask",
        "model": "multitask",
        "checkpoints": [
            "outputs/busi/multi/img_224/cbam/seg_weight_1/best_model.pt",
        ],
    },
    {
        "name": "mk_mnet_width025",
        "model": "mk_mnet",
        "width_mult": "0.25",
        "checkpoints": [str(CHECKPOINT_RELATIVE)],
    },
    {
        "name": "r_cbam_mnet",
        "model": "r_cbam_mnet",
        "checkpoints": [
            "outputs/busi/r_cbam_mnet/img_224/oversample_1/lambda_0.8/lr_0.0003/weight_decay_0.0001/patience_10/best_model.pt",
            "outputs/busi/r_cbam_mnet/img_224/seg_weight_1.2/lr_0.001/weight_decay_0.001/patience_100/best_joint.pt",
        ],
    },
]

completed = []
skipped = []
for run in MODEL_RUNS:
    checkpoint = next(
        (ARTIFACT_ROOT / path for path in run["checkpoints"] if (ARTIFACT_ROOT / path).is_file()),
        None,
    )
    if checkpoint is None:
        print(f"SKIP {run['name']}: không tìm thấy checkpoint")
        skipped.append(run["name"])
        continue

    output_dir = PROJECT_DIR / "gradcam-results" / run["name"] / "stage_5" / "target_true"
    command = [
        sys.executable, "generate_gradcam.py",
        "--model", run["model"],
        "--checkpoint", str(checkpoint),
        "--dataset-dir", str(DATASET_DIR),
        "--output-dir", str(output_dir),
        "--stage", "5",
        "--target-class", "true",
        "--all-samples",
        "--seed", "42",
    ]
    if "width_mult" in run:
        command.extend(["--width-mult", run["width_mult"]])

    print(f"\nRUN {run['name']}: {checkpoint}")
    subprocess.run(command, cwd=PROJECT_DIR, check=True)
    completed.append(run["name"])

print("\nHoàn thành:", completed)
print("Bỏ qua:", skipped)